## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is usefull for the following:

- Tracking agent behaviour with logging, analytics, and debugging.
- Transforming prompts, tool section, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")  # Set

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceeds context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased Summarization
agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "groq:qwen/qwen3-32b",
            trigger = ("messages", 10),
            keep = ("messages", 5)
        )
    ]
)

In [3]:
### Run with thread id
config = {"configurable": {"thread_id": "test-1"}}

In [5]:
### Alternative test data
questions = [
    "What is 2 + 2?",
    "What is 5 * 10?",
    "What is 100/4?",
    "What is 15 - 7?", 
    "What is 3 * 3?",
    "What is 50 + 25?",
    "What is 20 - 5?",
    "What is 4 * 4?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n<think>\nOkay, the user is asking for the speed of light. Let me start by recalling the basic facts. The speed of light in a vacuum is a fundamental constant, so I need to get the exact value right. I think it's 299,792,458 meters per second. That's approximately 300,000 km/s or 186,282 miles per second. I should mention that it's denoted by the letter 'c' in physics equations.\n\nWait, the user might be interested in the context where this speed is measured. I should note that it's the maximum speed at which all energy, matter, and information in the universe can travel. Also, Einstein's theory of relativity is closely tied to this speed. Maybe include that in the answer for context.\n\nThey might also want to know why the speed of light is important. Applications include technologies like GPS, which relies on accounting for relativistic effects. Or maybe mention that it's a key component i

### Token Size

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels - return long response to use more tokens"""
    return f"""Here are some {city}:
    1. Hotel A - A luxurious hotel located in the heart of {city}, offering stunning views and top-notch amenities. Perfect for travelers seeking comfort and elegance.
    2. Hotel B - A budget-friendly option in {city} that provides clean and comfortable accommodations. Ideal for travelers looking for value without compromising on quality.
    3. Hotel C - A boutique hotel in {city} known for its unique design and personalized service. Great for travelers who appreciate a more intimate and stylish atmosphere."""

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "groq:qwen/qwen3-32b",
            trigger = ("tokens", 250),
            keep = ("tokens", 100)
        )
    ]
)

config = {"configurable": {"thread_id": "test-2"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    # Approximate token count (1 token ≈ 4 characters)
    return total_chars // 4

In [7]:
# Run test2
cities = ["New York", "Paris", "Tokyo", "London", "Sydney"]

for city  in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

New York: ~321 tokens, 4 messages
[HumanMessage(content='Find hotels in New York', additional_kwargs={}, response_metadata={}, id='d624e76f-0ec0-4e3f-bb49-0e4c4019399d'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking to find hotels in New York. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The description says it returns a long response to use more tokens. Since the user wants hotels in New York, I should call this function with "New York" as the city. I need to make sure the arguments are correctly formatted in JSON. The required parameter is city, so I just need to provide that. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': '816why78x', 'function': {'arguments': '{"city":"New York"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 156, 'total_tokens': 287, 'completion_time': 0.188025

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels - return long response to use more tokens"""
    return f"""Here are some {city}:
    1. Hotel A - A luxurious hotel located in the heart of {city}, offering stunning views and top-notch amenities. Perfect for travelers seeking comfort and elegance.
    2. Hotel B - A budget-friendly option in {city} that provides clean and comfortable accommodations. Ideal for travelers looking for value without compromising on quality.
    3. Hotel C - A boutique hotel in {city} known for its unique design and personalized service. Great for travelers who appreciate a more intimate and stylish atmosphere."""

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "groq:qwen/qwen3-32b",
            trigger = ("fraction", 0.005),  # Trigger when messages exceed 0.5% of context window
            keep = ("fraction", 0.002)  # Keep the most recent 0.2% of tokens in messages
        )
    ]
)

config = {"configurable": {"thread_id": "test-3"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    # Approximate token count (1 token ≈ 4 characters)
    return total_chars // 4

# Run test-3
cities = ["New York", "Paris", "Tokyo", "London", "Sydney"]

for city  in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

### Human in the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Simulate reading an email by ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Simulate sending an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body '{body}'"

In [12]:
agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool": {
                    "allowed_decisions": ["approve", "modify", "reject"],
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [13]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'Hi John, how are you?'")]},
    config = config
)

In [14]:
result 

{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'Hi John, how are you?'", additional_kwargs={}, response_metadata={}, id='c8272f06-b653-4287-b44d-600f7f50c6cb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@example.com with the subject 'Hello' and the body 'Hi John, how are you?'. Let me check the available tools.\n\nLooking at the tools provided, there's a send_email_tool. Its parameters are recipient, subject, body, all required. The function requires those three fields. The user provided all three: recipient is john@example.com, subject is 'Hello', and the body is 'Hi John, how are you?'. \n\nSo I need to call the send_email_tool with these arguments. I don't need to use the read_email_tool here because the request is about sending, not reading. All the necessary parameters are present, so I can proceed to make the tool call.\n", 'tool_calls': [{'id': 'rsma3chmz', 'func

In [15]:
from langgraph.types import Command
# Step 2 : Approve
if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config = config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: The email has been successfully sent to **john@example.com** with the subject **"Hello"** and the body:  
*"Hi John, how are you?"*  

Let me know if you need anything else!


In [16]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'Hi John, how are you?'", additional_kwargs={}, response_metadata={}, id='c8272f06-b653-4287-b44d-600f7f50c6cb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@example.com with the subject 'Hello' and the body 'Hi John, how are you?'. Let me check the available tools.\n\nLooking at the tools provided, there's a send_email_tool. Its parameters are recipient, subject, body, all required. The function requires those three fields. The user provided all three: recipient is john@example.com, subject is 'Hello', and the body is 'Hi John, how are you?'. \n\nSo I need to call the send_email_tool with these arguments. I don't need to use the read_email_tool here because the request is about sending, not reading. All the necessary parameters are present, so I can proceed to make the tool call.\n", 'tool_calls': [{'id': 'rsma3chmz', 'func

### Reject

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Simulate reading an email by ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Simulate sending an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body '{body}'"

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool": {
                    "allowed_decisions": ["approve", "modify", "reject"],
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [18]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'Hi John, how are you?'")]},
    config = config
)

In [19]:
from langgraph.types import Command
# Step 2 : Reject
if "__interrupt__" in result:
    print("Paused! Rejecting...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config = config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Rejecting...
Result: It seems the email couldn't be sent at this time (tool call rejected with ID etqawj6y7). Would you like me to try again or help with a different request?


In [20]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'Hi John, how are you?'", additional_kwargs={}, response_metadata={}, id='860e34a1-264f-4fad-ba79-5d904957811a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@example.com with the subject 'Hello' and the body 'Hi John, how are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are provided here, so I should use that tool. The parameters are all required, so I just need to structure the JSON with those fields. No need for the read_email_tool since the user isn't asking to read an email. Alright, I'll format the tool call with the given parameters.\n", 'tool_calls': [{'id': 'etqawj6y7', 'function': {'arguments': '{"body":"Hi John, how are you?","recipient":"john@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_m

### Modify

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Simulate reading an email by ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Simulate sending an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body '{body}'"

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools = [read_email_tool, send_email_tool],
    checkpointer = InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool": {
                    "allowed_decisions": ["approve", "modify", "reject"],
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [22]:
config = {"configurable": {"thread_id": "test-modify"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@example.com with subject 'Hello' and body 'Hi John, how are you?'")]},
    config = config
)

In [24]:
result

{'messages': [HumanMessage(content="Send email to wrong@example.com with subject 'Hello' and body 'Hi John, how are you?'", additional_kwargs={}, response_metadata={}, id='b3566cdf-975d-4122-a661-a2df46a4a1a5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to wrong@example.com with the subject \'Hello\' and the body \'Hi John, how are you?\'. Let me check the available tools. There\'s a send_email_tool that requires recipient, subject, and body. The parameters are all required, so I need to include them. The function call should have the name "send_email_tool" and the arguments with the provided details. I\'ll structure the JSON accordingly.\n', 'tool_calls': [{'id': 'j8ndw67fp', 'function': {'arguments': '{"body":"Hi John, how are you?","recipient":"wrong@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 251, 'total_

In [ ]:
from langgraph.types import Command
# Step 2 : Modify and Approve
if "__interrupt__" in result:
    print("Paused! Modifying...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions": [
                    {"type": "modify",
                    "edited_action": {
                        "name" : "send_email_tool",
                        "args": {
                            "recipient": "john@example.com",
                            "subject": "Corrected Subject",
                            "body": "This was modified by human in the loop middleware before sending"
                        }
                    }}
                ]
            }
        ),
        config = config
    )
